Data Analysis and Machine Learning

Fynn Wäller 28.08.2026

# Explorative Analyse von Grand-Slam-Matches im Tennis (2000–2026)

<a id="toc"></a>

1. [Aufgabe und Ziele](#kap1)
   1. [Abweichungen vom Proposal](#kap1-1)
2. [Daten verstehen und aufbereiten](#kap2)
   1. [Daten einlesen und erster Überblick](#kap2-1)
   2. [Strukturprüfung](#kap2-2)
   3. [Kontrolle und Platzhalter](#kap2-3)
   4. [Bereinigung und Verifikation der Punkte-Spalten](#kap2-4)
   5. [Ableitung der Hilfsspalten](#kap2-5)
3. [Explorative Datenanalyse](#kap3)
   1. [Rangdifferenz und Siegwahrscheinlichkeit (H1)](#kap3-1)
   2. [Favoritensiege nach Belag und Chi-Quadrat-Test (H2)](#kap3-2)
   3. [Vorhersagbarkeit im Zeitverlauf (H3)](#kap3-3)
   4. [Top-10-Dominanz (H4)](#kap3-4)
   5. [Prädiktorvergleich: Weltrangliste vs. Buchmacherquote (H5)](#kap3-5)
4. [Vorhersage des Matchausgangs](#kap4)
5. [Fazit und Limitationen](#kap5)


<a id="kap1"></a>

## 1. Aufgabe und Ziele

[&#8593; Zurück zum Inhaltsverzeichnis](#toc)

Grand-Slam-Turniere gehören zu den wichtigsten Wettbewerben im Tennis. Ziel dieses Projekts ist es, anhand aller Grand-Slam-Matches seit 2000 zu untersuchen, welche Faktoren den Ausgang einzelner Matches beeinflussen, insbesondere die Weltranglistenposition, der Belag und die zeitliche Entwicklung der Vorhersagbarkeit eines Matches.

Datengrundlage ist der Kaggle-Datensatz "ATP Tennis 2000–2026 Daily Update" (dissfya) mit allen ATP-Matches seit 2000 (Stand 19.07.2026).
Quelle: <https://www.kaggle.com/datasets/dissfya/atp-tennis-2000-2023daily-pull>

Der Datensatz liegt dieser Abgabe als `atp_tennis_dataset.csv` bei, außerdem existiert eine Beschreibung aller Spalten als `atp_tennis_dataset_INFO.csv`. Für die Analyse wird der Datensatz auf Grand-Slam-Turniere gefiltert, hierbei werden alle Runden mit einbezogen.

Untersucht werden fünf Hypothesen:

- **H1 – Rangdifferenz:** Je größer der Rangunterschied zwischen zwei Spielern, desto höher die Siegwahrscheinlichkeit des besser platzierten Spielers.
- **H2 – Belag:** Auf Sand gewinnen Favoriten seltener als auf Hartplatz oder Rasen, da der Belag die Vorhersagbarkeit eines Matches beeinflusst.
- **H3 – Zeitverlauf:** Die Favoritensiegquote ist im Zeitverlauf gestiegen, insbesondere während der "Big-Three-Ära" waren Grand-Slam-Ergebnisse vorhersagbarer als davor.
- **H4 – Top-10-Dominanz:** Top-10-platzierte Spieler gewinnen überproportional viele Matches gegen Spieler außerhalb der Top 10.
- **H5 – Prädiktorvergleich:** Die Buchmacherquote sagt den Matchausgang besser vorher als die Weltranglistenposition, da sie Form, Verletzungen und Head-to-Head beinhaltet.


<a id="kap1-1"></a>

### 1.1 Abweichungen vom Proposal

Die fünfte Hypothese wurde gegenüber dem Proposal angepasst. Ursprünglich sollte die Differenz der ATP-Punkte gegen die Rangdifferenz als Prädiktor getestet werden. Beide Größen haben jedoch dieselbe Information, da die Weltrangliste direkt aus den ATP-Punkten gebildet wird, sie benennen in jedem einzelnen auswertbaren Match denselben Favoriten, sodass sich kein Unterschied messen lässt. Die Buchmacherquote weicht dagegen mehr vom Rang-Favoriten ab und ist damit der aussagekräftigere Vergleichsmaßstab. (Dazu mehr in Abschnitt 3.5.)

Mit dieser Änderung entfällt auch die im Proposal geplante Hilfsspalte für die Punktedifferenz. Der Punkte-Favorit wird in Abschnitt 3.5 trotzdem gebildet, dort dient er dem Nachweis, dass Rang und Punkte denselben Favoriten benennen.

### Ergänzung: Vorhersagemodell

Vorhergesagt wird, ob der besser platzierte Spieler gewinnt. Als Verfahren dient eine logistische Regression auf einem Train-Test-Split, gemessen wird gegen zwei Maßstäbe: nach unten die triviale Regel "immer gewinnt der Rang-Favorit", nach oben den Quoten-Favoriten der Buchmacher.

### Verwendete Bibliotheken

Das Notebook nutzt pandas, numpy, matplotlib, scipy und scikit-learn. Alle Abhängigkeiten werden zu Beginn des Notebooks importiert und ggf. installiert. 

In [ ]:
# Fehlende Abhängigkeiten zu Beginn installiert, falls sie fehlen, und importiert:
%pip install -q pandas numpy matplotlib scipy scikit-learn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

<a id="kap2"></a>

## 2. Daten verstehen und aufbereiten

[&#8593; Zurück zum Inhaltsverzeichnis](#toc)

<a id="kap2-1"></a>

### 2.1 Daten einlesen und erster Überblick

Die Daten werden in ein DataFrame geladen. Ein erster Überblick zeigt den Gesamtumfang, die enthaltenen Turnierserien und die Anzahl der Grand-Slam-Matches:

In [ ]:
df = pd.read_csv("atp_tennis_dataset.csv")
df.head(1)

Zusätzlich liegt eine ausgelagerte INFO-Datei bei, die alle Spalten erklärt. Hier stehen zunächst nur die Rohdatenspalten, die selbst erzeugten Hilfsspalten folgen in Abschnitt 2.5.

In [ ]:
# Spaltenbeschreibung aus der beiliegenden INFO-Datei, Semikolon als Trennzeichen
stats_info = pd.read_csv("atp_tennis_dataset_INFO.csv", sep=";")

# Ohne diese Zeile schneidet pandas die Beschreibungen bei 50 Zeichen ab
pd.set_option('display.max_colwidth', None)

# Ab dieser Zeile stehen die Spalten, die erst im Notebook entstehen. Sie folgen in 2.5.
grenze = stats_info.index[stats_info['columns'] == "ABGELEITETE SPALTEN"][0]

stats_info.iloc[:grenze].dropna()

In [ ]:
print("Alle ATP-Matches:", len(df))

print(df['Series'].unique())

print("Grand-Slam-Matches:", len(df[df['Series'] == 'Grand Slam']))

Der Datensatz umfasst 68.221 ATP-Matches, davon 12.871 in der Turnierserie "Grand Slam". Die Serienbezeichnungen haben sich bei anderen Turnierarten im Zeitverlauf verändert (z. B. "Masters" wurde zu "Masters 1000"); für dieses Projekt ist nur der über alle Jahre einheitliche Wert "Grand Slam" relevant.

<a id="kap2-2"></a>

### 2.2 Strukturprüfung

Als Nächstes werden Spaltennamen, Datentypen und fehlende Werte geprüft:

In [ ]:
df.info()
len(df)

Alle 17 Spalten sind vollständig befüllt (keine NaN), zehn Spalten sind Strings, sieben numerisch. Fehlende Werte sind demnach nicht als NaN, sondern vermutlich als Platzhalter verzeichnet, was im nächsten Schritt untersucht wird.

<a id="kap2-3"></a>

### 2.3 Kontrolle und Platzhalter

Verteilungen und Minima der numerischen Spalten zeigen, wie fehlende Werte kodiert sind und ob die Pts-Spalten tatsächlich ATP-Ranglistenpunkte enthalten (Die ursprüngliche Erwartung: guter Rang <-> viele Punkte):

In [ ]:
# Exakten Grand-Slam-Wert finden
print(df['Series'].unique())

# Platzhalter-Check: Minima und verdächtige Werte
print(df[['Rank_1','Rank_2','Pts_1','Pts_2','Odd_1','Odd_2']].describe())
print("Rank <= 0:", (df['Rank_1'] <= 0).sum() + (df['Rank_2'] <= 0).sum())
print("Pts <= 0:", (df['Pts_1'] <= 0).sum() + (df['Pts_2'] <= 0).sum())

# Spearman, damit der Wert direkt mit dem nach der Bereinigung vergleichbar ist
print(df[['Rank_1','Pts_1']].corr(method='spearman'))   # erwartet: negativ (Rang 1 = beste Position)

#Grand-Slam-Filter (Wert aus Schritt 1 einsetzen) und zählen
df_gs = df[df['Series'] == 'Grand Slam']
print("Grand-Slam-Matches:", len(df_gs))
print(df_gs['Round'].unique())

Die Minima von −1 in den Rank-, Pts- und Odd-Spalten sind Platzhalter für fehlende Werte. Betroffen sind 26 Ranglistenplätze.

Die Spearman-Korrelation von −0,59 zwischen Rang und Punkten deutet bereits darauf hin, dass es sich bei Pts um ATP-Ranglistenpunkte handelt. Der Wert ist allerdings noch durch die Platzhalter verzerrt: Eine −1 gilt beim Rang als bestmögliche Position und bei den Punkten zugleich als niedrigster Wert. Nach der Bereinigung im nächsten Schritt wird derselbe Wert erneut berechnet und ist dann direkt vergleichbar.



<a id="kap2-4"></a>

### 2.4 Bereinigung und Verifikation der Punkte-Spalten

Die Platzhalter werden in echte NaN umgewandelt, außerdem wird eine Jahresspalte abgeleitet und geprüft, in welchen Jahren die Punkte fehlen:

In [ ]:
# Datumsformat kurz pruefen, die Jahresspalte wird gleich daraus abgeleitet
print(df['Date'].head(3))

In [ ]:
for col in ['Rank_1','Rank_2','Pts_1','Pts_2','Odd_1','Odd_2']:
    df[col] = df[col].replace(-1, np.nan)

print("Pts == 0:", (df[['Pts_1','Pts_2']] == 0).sum().sum())

df['Year'] = df['Date'].str[:4].astype(int)
df_gs = df[df['Series'] == 'Grand Slam'].copy()

# Anteil fehlender Punkte pro Jahr (in %)
fehlend = df_gs.groupby('Year')['Pts_1'].apply(lambda s: round(s.isna().mean()*100, 1))
print(fehlend)

# Saubere Pts-Verifikation (Spearman, nur gültige Werte)
clean = df_gs.dropna(subset=['Rank_1','Pts_1'])
print(clean[['Rank_1','Pts_1']].corr(method='spearman'))

Ergebnis der Bereinigung: `-1` ist der einzige Platzhalter, 0 kommt nicht vor. Die ATP-Punkte fehlen 2000–2004 vollständig und 2005 zu rund 75 %, ab 2006 sind die Einträge lückenlos. Die Spearman-Korrelation steigt von −0,59 vor der Bereinigung auf −0,98 danach und bestätigt die Interpretation als Ranglistenpunkte. Der Sprung geht allein auf die entfernten Platzhalter zurück, gerechnet wurde beide Male mit demselben Verfahren. 

Zum Abschluss ein Blick auf die Verteilungen aller numerischen Spalten.

Ränge und Punkte sind stark rechtsschief, rund 79 % aller Spieler stehen in den Top 100, einzelne reichen aber bis über Rang 1.800. Auffällig ist zudem, wie ähnlich sich Spieler 1 und Spieler 2 sehen, beide mit demselben Median von 50. Das bestätigt, dass die Reihenfolge der beiden Spalten keine Bedeutung trägt, ein Punkt, der in Kapitel 4 für die Merkmalsbildung wichtig wird.

Zwei Panels dienen der Vollständigkeitsprüfung: `Best of` zeigt die 23 Matches, die fälschlich mit drei statt fünf Sätzen kodiert sind, und `Year` zeigt rund 485 Matches pro Jahr mit den beiden erkennbaren Ausreißern 2020 und 2026.

In [ ]:
# Verteilungen aller numerischen Spalten, die in df_gs vorkommen
df_gs.select_dtypes('number').hist(bins=40, figsize=(12, 7), color='#4a7ebb')
plt.tight_layout()
plt.show()

<a id="kap2-5"></a>

### 2.5 Ableitung der Hilfsspalten

Die Rohdaten enthalten pro Match beide Spieler, ihre Ränge, die Buchmacherquoten und den Sieger. Was fehlt, ist eine Spalte, die den Favoriten benennt und festhält, ob er gewonnen hat. Genau diese Information brauchen alle fünf Hypothesen.



| Spalte | Bedeutung | Benötigt für |
|---|---|---|
| `Rang_Diff` | Betrag der Rangdifferenz beider Spieler | H1 |
| `Fav_Rang` | Spieler mit dem besseren (niedrigeren) Rang | H1–H3 |
| `Fav_Rang_Sieg` | Hat der Rang-Favorit gewonnen? | H1–H3 |
| `Fav_Quote` | Spieler mit der niedrigeren Buchmacherquote | H5 |
| `Fav_Quote_Sieg` | Hat der Quoten-Favorit gewonnen? | H5 |
| `Top10_1`, `Top10_2` | Steht der jeweilige Spieler in den Top 10? | H4 |
| `Top10_Sieg` | Hat der Top-10-Spieler gegen einen Nicht-Top-10-Spieler gewonnen? | H4 |

Weitere Spalten entstehen erst dort, wo sie gebraucht werden:

| Spalte | Bedeutung | Entsteht in |
|---|---|---|
| `Year` | Jahr des Matches, aus `Date` abgeleitet | 2.4 |
| `Rang_Klasse` | `Rang_Diff` in sechs Klassen: 1-5, 6-10, 11-20, 21-50, 51-100, 100+ | 3.1 |
| `Aera` | Zeitraum des Matches: vor 2004, 2004-2022 oder ab 2023 | 3.3 |
| `Top10_Duell` | True, wenn genau einer der beiden Spieler in den Top 10 steht | 3.4 |
| `Fav_Pts` | Spieler mit den meisten ATP-Punkten, nur für den Nachweis in 3.5 | 3.5 |
| `Rang_Fav`, `Top10_Fav`, `Top10_Beide` | Symmetrische Merkmale für das Vorhersagemodell | 4 |



Zwei Fälle müssen dabei bewusst betrachtet werden, weil ein einfacher Vergleich mit `<` sie sonst falsch zuordnen würde.


1. **Fehlende Werte:** Ist ein Rang oder eine Quote `NaN`, ergibt jeder Vergleich `False`, der zweite Spieler würde automatisch zum Favoriten erklärt. Solche Matches erhalten stattdessen `NaN` und fallen aus der jeweiligen Auswertung heraus.
2. **Gleichstände:** Sind beide Werte identisch, gibt es keinen Favoriten. Auch diese Matches werden auf `NaN` gesetzt.


Durch die Verwendung des Datentyps `boolean` (nullable) bleiben diese Lücken erhalten, statt fälschlich als `False` gezählt zu werden. Damit rechnet `.mean()`(Durchschnitt) automatisch nur über die gültigen Fälle.


Zunächst wird die Rangdifferenz sowie die Favoriten-Kennzeichen nach Rang und nach Quote ermittelt:

In [ ]:
# Rangdifferenz als Betrag, damit die Reihenfolge von Player_1/Player_2 keine Rolle spielt
df_gs['Rang_Diff'] = (df_gs['Rank_1'] - df_gs['Rank_2']).abs()

# Favorit nach Rang ist  nur gueltig, wenn beide Ränge vorliegen und nicht gleich sind
rang_ok = df_gs[['Rank_1','Rank_2']].notna().all(axis=1) & (df_gs['Rank_1'] != df_gs['Rank_2'])
fav_rang = pd.Series(np.where(df_gs['Rank_1'] < df_gs['Rank_2'],   # niedriger = besser
                              df_gs['Player_1'], df_gs['Player_2']), index=df_gs.index)
df_gs['Fav_Rang'] = fav_rang.where(rang_ok)
df_gs['Fav_Rang_Sieg'] = (df_gs['Fav_Rang'] == df_gs['Winner']).where(rang_ok).astype('boolean')

In [ ]:
# Favorit nach Buchmacherquote, mit der gleichen Logik (niedrige Quote = Favorit)
quote_ok = df_gs[['Odd_1','Odd_2']].notna().all(axis=1) & (df_gs['Odd_1'] != df_gs['Odd_2'])
fav_quote = pd.Series(np.where(df_gs['Odd_1'] < df_gs['Odd_2'],
                               df_gs['Player_1'], df_gs['Player_2']), index=df_gs.index)
df_gs['Fav_Quote'] = fav_quote.where(quote_ok)
df_gs['Fav_Quote_Sieg'] = (df_gs['Fav_Quote'] == df_gs['Winner']).where(quote_ok).astype('boolean')

Für H4 wird zusätzlich festgehalten, welcher Spieler in den Top 10 steht. Aussagekräftig sind dabei nur Matches, in denen **genau einer** der beiden Spieler zu den Top 10 gehört, das heißt, wenn zwei Top-10-Spieler oder zwei Außenseiter aufeinandertreffen, gibt es keinen Vergleich. Das exklusive Oder (`^`) filtert genau diese Matches heraus:

In [ ]:
# Steht der jeweilige Spieler in den Top 10?
# Wenn eine Zeile einen fehlenden Rang hat, wird das Ergebnis auf NaN gesetzt (daher die where()-Funktion)
df_gs['Top10_1'] = (df_gs['Rank_1'] <= 10).where(df_gs['Rank_1'].notna()).astype('boolean')
df_gs['Top10_2'] = (df_gs['Rank_2'] <= 10).where(df_gs['Rank_2'].notna()).astype('boolean')

# Nur Matches, in denen genau einer der beiden Spieler Top 10 ist
duell = (df_gs['Top10_1'] ^ df_gs['Top10_2']).fillna(False)
top10_spieler = pd.Series(np.where(df_gs['Top10_1'].fillna(False),
                                   df_gs['Player_1'], df_gs['Player_2']), index=df_gs.index)
df_gs['Top10_Sieg'] = (top10_spieler == df_gs['Winner']).where(duell).astype('boolean')

Damit sind alle Hilfsspalten angelegt. Für die Nachvollziehbarkeit der späteren Auswertungen wird dokumentiert, wie viele Matches je Hypothese zur Verfügung stehen und wie viele ausgeschlossen werden mussten:

In [ ]:
print("Grand-Slam-Matches gesamt:         ", len(df_gs))
print("davon mit gueltigem Rang-Favorit:  ", int(rang_ok.sum()),
      "| ausgeschlossen:", int((~rang_ok).sum()))
print("davon mit gueltigem Quoten-Favorit:", int(quote_ok.sum()),
      "| ausgeschlossen:", int((~quote_ok).sum()),
      "(davon", int((df_gs['Odd_1'] == df_gs['Odd_2']).sum()), "Gleichstaende)")
print("Duelle Top 10 gegen Nicht-Top-10:  ", int(duell.sum()))

Zur Datenbasis der einzelnen Hypothesen:

- **Rang-Favorit:** In allen 12.871 Grand-Slam-Matches sind beide Ränge vorhanden, und es gibt keinen einzigen Rang-Gleichstand. H1 bis H4 stehen also auf der vollen Datenbasis, die 26 ungültigen Ränge aus Abschnitt 2.3 betreffen ausschließlich Matches außerhalb der Grand Slams.
- **Quoten-Favorit:** Hier bleiben 12.089 Matches (94 %). Ausgeschlossen werden 608 Matches ohne Quote, diese Zeilen stammen überwiegend aus den frühen 2000er Jahren, in denen der Datensatz noch keine Quoten enthält, sowie 174 Matches mit identischen Quoten für beide Spieler, in denen es rechnerisch keinen Favoriten gibt.
- **Top-10-Duelle:** In 3.406 Matches steht genau ein Top-10-Spieler einem Spieler außerhalb der Top 10 gegenüber. Nur diese Matches sind für H4 aussagekräftig.

Als abschließende Kontrolle werden die Siegquoten berechnet und eine Stichprobe der neuen Spalten ausgegeben. Wären die Hilfsspalten fehlerhaft gebildet, müssten die Quoten auffällig nahe bei 50 % (zufällige Zuordnung) oder nahe bei 100 % liegen:

In [ ]:
print("Favoritensiegquote nach Rang:       ", round(df_gs['Fav_Rang_Sieg'].mean()*100, 2), "%")
print("Favoritensiegquote nach Quote:      ", round(df_gs['Fav_Quote_Sieg'].mean()*100, 2), "%")
print("Siegquote Top 10 gegen Nicht-Top-10:", round(df_gs['Top10_Sieg'].mean()*100, 2), "%")

df_gs[['Player_1','Player_2','Rank_1','Rank_2','Rang_Diff',
       'Fav_Rang','Winner','Fav_Rang_Sieg']].head()

Alle drei Werte liegen im plausiblen Bereich, und die Stichprobe bestätigt die Zuordnung: In der ersten Zeile ist Alami K. mit Rang 35 gegenüber Manta L. der Favorit und gewinnt auch, dabei steht `Fav_Rang_Sieg` korrekt auf `True`.

Inhaltlich zeichnet sich bereits die Richtung der späteren Ergebnisse ab: Der Rang-Favorit gewinnt 71,53 % seiner Matches, der Quoten-Favorit 76,64 %, was ein Vorsprung für die Buchmacher ist (H5). Trifft ein Top-10-Spieler auf einen Gegner außerhalb der Top 10, gewinnt er in 86,02 % der Fälle (H4).

Die beiden Favoritenquoten beruhen dabei auf unterschiedlich vielen Matches, weil nicht in jedem Match eine Quote vorliegt. Der faire Vergleich auf gemeinsamer Basis folgt in 3.5.

Diese Gesamtwerte werden in Kapitel 3 differenziert nach Rangdifferenz, Belag und Jahr untersucht.

<a id="kap3"></a>

## 3. Explorative Datenanalyse

[&#8593; Zurück zum Inhaltsverzeichnis](#toc)

Die fünf Hypothesen werden nacheinander untersucht. Grundlage ist jeweils das bereinigte Grand-Slam-DataFrame `df_gs`. Jede Auswertung nennt die verwendete Anzahl an Matches (Fallzahl) und dokumentiert ausgeschlossene Matches.

Vier der fünf Hypothesen laufen auf dieselbe Rechnung hinaus: eine Siegquote je Gruppe, viermal nur mit anderem Gruppierungsmerkmal.

- Rangdifferenz (H1)
- Belag (H2)
- Jahr (H3)
- Turnierrunde (H4)

Dafür wird einmalig eine Hilfsfunktion definiert, die neben der Quote immer auch die Anzahl der verwendbaren Matches zurückgibt:

In [ ]:
# Funktion zur Berechnung der Favoritensiegquote je Gruppe
def quote_je_gruppe(daten, gruppe, ziel="Fav_Rang_Sieg"):
    """Favoritensiegquote in Prozent und Fallzahl je Gruppe."""
    t = daten.groupby(gruppe, observed=True)[ziel].agg(Quote="mean", Matches="size")
    t["Quote"] = (t["Quote"] * 100).round(2)
    return t

<a id="kap3-1"></a>

### 3.1 Rangdifferenz und Siegwahrscheinlichkeit (H1)

Geprüft wird, ob die Siegwahrscheinlichkeit des besser platzierten Spielers mit wachsendem Rangunterschied steigt.

Eine Auswertung je einzelnem Differenzwert wäre dafür nicht sinnvoll, da die Rangdifferenz von 1 bis fast 1.800 reicht und die meisten Einzelwerte nur in einer Handvoll Matches vorkämen. Die Matches werden deshalb in sechs "Container-Klassen" zusammengefasst:

**1–5 · 6–10 · 11–20 · 21–50 · 51–100 · über 100**

Die Stufen sind unten bewusst fein und oben grob gewählt. Im Bereich kleiner Rangunterschiede liegt die eigentlich interessante Frage: 
ab welchem Abstand trifft die Weltrangliste überhaupt eine belastbare Aussage? Oberhalb von 100 Rängen ist eine feinere Unterteilung wenig aufschlussreich, weil dort ohnehin ein deutlicher Klassenunterschied zwischen den Spielern besteht.

Für jede Klasse werden die Favoritensiegquote und die Fallzahl berechnet. Die Fallzahl gehört zwingend dazu, weil sich eine Quote ohne ihre Grundlage nicht einordnen lässt.

In [ ]:
# pd.cut schliesst die linke Grenze aus: (0, 5] entspricht einer Rangdifferenz von 1 bis 5.
# Rang_Diff == 0 kommt in den Grand Slams nicht vor, es geht also kein Match verloren.
grenzen = [0, 5, 10, 20, 50, 100, np.inf]
labels  = ["1-5", "6-10", "11-20", "21-50", "51-100", "100+"]
df_gs['Rang_Klasse'] = pd.cut(df_gs['Rang_Diff'], bins=grenzen, labels=labels)

h1 = quote_je_gruppe(df_gs, 'Rang_Klasse')
print(h1)
print("\nSumme der Fallzahlen:", int(h1['Matches'].sum()), "von", len(df_gs), "Matches")

In [ ]:
gesamt = df_gs['Fav_Rang_Sieg'].mean() * 100

plt.figure(figsize=(8, 5))
plt.plot(h1.index.astype(str), h1['Quote'], marker='o', lw=2, label='Favoritensiegquote')
plt.axhline(50, ls='--', color='grey', lw=1, label='Münzwurf (50 %)')
plt.axhline(gesamt, ls=':', color='darkred', lw=1.5, label=f'Gesamtquote ({gesamt:.1f} %)')
plt.xlabel("Rangdifferenz zwischen den Spielern")
plt.ylabel("Favoritensiegquote in %")
plt.title("Favoritensiegquote je Rangdifferenz-Klasse")
plt.ylim(45, 85)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

Zur Absicherung wird die Quote über einen zweiten, unabhängigen Rechenweg bestimmt:
Diesmal nicht über den Namen des Favoriten, sondern über den Rang des Siegers. Beide Wege müssen dasselbe Ergebnis liefern, tun sie es nicht, würde ein Fehler in der Ableitung der Hilfsspalten aus Abschnitt 2.5 vorliegen. 

In [ ]:
# Zweiter Rechenweg: Rang des Siegers mit dem des Verlierers vergleichen
sieger_ist_p1 = df_gs['Winner'] == df_gs['Player_1']
fav_gewann    = np.where(sieger_ist_p1, df_gs['Rank_1'] < df_gs['Rank_2'],
                                        df_gs['Rank_2'] < df_gs['Rank_1'])

print("Quote über Fav_Rang_Sieg aus 2.5 :", round(df_gs['Fav_Rang_Sieg'].mean() * 100, 2), "%")
print("Quote über den Rang des Siegers  :", round(fav_gewann.mean() * 100, 2), "%")
print("Matches mit abweichendem Ergebnis:", int((fav_gewann != df_gs['Fav_Rang_Sieg']).sum()))

Beide Rechenwege kommen auf dieselbe Quote, und zwar in jedem einzelnen der 12.871 Matches. Die Hilfsspalten aus 2.5 sind damit korrekt abgeleitet. Die Quote selbst steigt über alle sechs Klassen hinweg, ohne einen einzigen Rückschritt.


**H1 ist damit bestätigt.** Je größer der Rangunterschied, desto häufiger gewinnt der besser platzierte Spieler, von 57,16 % bei sehr eng benachbarten Spielern bis 79,71 % bei mehr als 100 Rängen Abstand.

Aufschlussreicher als die Bestätigung selbst ist der Verlauf der Quote über die Klassen:

- **Bei kleinen Rangunterschieden ist der Favoritenstatus fast wertlos.** Mit 57 % liegen die beiden untersten Klassen nur knapp über dem Münzwurf. Zwischen Rang 20 und Rang 25 trifft die Weltrangliste praktisch keine belastbare Aussage darüber, wer gewinnt.
- **Die Kernaussage steckt im mittleren Bereich.** Zwischen den Klassen 11–20 und 51–100 springt die Quote von 62,87 % auf 76,91 %, hier entscheidet sich, ob ein Rangunterschied überhaupt aussagekräftig ist. Oberhalb von 100 Rängen kommen nur noch knapp drei Prozentpunkte hinzu. 
- **Einschränkung:** Die Klassen mitteln über die gesamte Rangliste hinweg. Eine Differenz von zehn Plätzen bedeutet an der Spitze (Rang 1 gegen Rang 11) etwas anderes als im Mittelfeld (Rang 150 gegen Rang 160), weil die Leistungsdichte oben deutlich höher ist. Die Ergebnisse sind daher als Durchschnitt über beide Situationen zu verstehen.

<a id="kap3-2"></a>

### 3.2 Favoritensiege nach Belag und Chi-Quadrat-Test (H2)

Geprüft wird, ob der Belag,  wie zuverlässig sich ein Match vorhersagen lässt. Gespielt wird auf Hartplatz, Sand und Rasen. Die Hypothese erwartet Sand als Ausreißer nach unten, weil das langsame Spiel einem Außenseiter mehr Gelegenheiten gibt, ein Match noch zu drehen.


In [ ]:
h2 = quote_je_gruppe(df_gs, 'Surface')
print(h2)
print("\nSumme der Fallzahlen:", int(h2['Matches'].sum()), "von", len(df_gs), "Matches")

# Hallen-Effekt als Stoergroesse ausschliessen
print("Court-Werte in den Grand Slams:", df_gs['Court'].unique())

In [ ]:
gesamt = df_gs['Fav_Rang_Sieg'].mean() * 100

plt.figure(figsize=(8, 5))
balken = plt.bar(h2.index, h2['Quote'], width=0.6,
                 color=['#c98a5e', '#5fa855', '#4a7ebb'])   # Sand, Rasen, Hartplatz
plt.axhline(gesamt, ls=':', color='darkred', lw=1.5, label=f'Gesamtquote ({gesamt:.1f} %)')

# Fallzahl in jeden Balken schreiben
for b, n in zip(balken, h2['Matches']):
    plt.text(b.get_x() + b.get_width() / 2, 61, f"n={n}", ha='center', color='white')

plt.xlabel("Belag")
plt.ylabel("Favoritensiegquote in %")
plt.title("Favoritensiegquote je Belag")
plt.ylim(60, 80)   # Ausschnitt, siehe Hinweis im Text
plt.grid(axis='y', alpha=0.3)
plt.legend()
plt.show()

Die y-Achse ist auf 60 bis 80 % zugeschnitten, das vergrößert die Unterschiede optisch und lässt sich besser interpretieren. Tatsächlich beträgt die Spannweite nur 2,58 Prozentpunkte.

Genau deshalb reicht die Tabelle als Beleg nicht aus. Drei Gruppen fallen auch dann nie exakt gleich aus, wenn der Belag überhaupt keine Rolle spielt, ein kleiner Unterschied entsteht immer durch Zufall. Der Chi-Quadrat-Test prüft, ob der beobachtete Unterschied größer ist als das, was der Zufall erklärt. Die Nullhypothese wäre dann: Der Belag und der Matchausgang sind unabhängig voneinander.

Mit ausgegeben werden die kleinste erwartete Häufigkeit (Voraussetzung des Tests: mindestens 5) und Cramérs V. Der p-Wert sagt nur, ob ein Zusammenhang besteht (Statistische Signifikanz) und Cramérs V sagt, wie stark er tatsächlich ist und das auf einer Skala von 0 bis 1. Als grobe Orientierung gilt ab 0,1 ein schwacher, ab 0,3 ein mittlerer und ab 0,5 ein starker Zusammenhang.

In [ ]:
# Anzahl der Matches je Belag und Favoritensieg
Tabelle = pd.crosstab(df_gs['Surface'], df_gs['Fav_Rang_Sieg'])
print(Tabelle)

chi2, p, dof, erwartet = chi2_contingency(Tabelle)
print("\nChi-Quadrat:", round(chi2, 3), "| p-Wert:", round(p, 5), "| Freiheitsgrade:", dof)
print("Kleinste erwartete Haeufigkeit:", round(erwartet.min(), 1), "(Voraussetzung: >= 5)")

# Cramers V: bei einer 3x2-Tafel ist min(Zeilen-1, Spalten-1) = 1,
# deshalb genuegt die verkuerzte Formel sqrt(chi2 / n).
v = np.sqrt(chi2 / Tabelle.values.sum())
print("Cramers V:", round(v, 4), "(0 = kein Zusammenhang, 1 = perfekter Zusammenhang)")

Drei Aussagen, die getrennt gehören:

- **Signifikant.** p = 0,031 liegt unter 0,05, die Nullhypothese wird daher verworfen. Die Voraussetzung ist mit 906,4 klar erfüllt.

- **Aber praktisch bedeutungslos.** Cramérs V liegt bei 0,023 auf einer Skala bis 1. Bei 12.871 Matches wird auch ein winziger Unterschied signifikant, weil der p-Wert allein durch die Datenmenge sinkt. Er misst, wie sicher ein Effekt existiert, nicht wie groß dieser ist.

- **Die Richtung stimmt nicht.** Sand liegt mit 71,51 % im Mittelfeld, den niedrigsten Wert hat aber Rasen mit 69,81 %. Auf Rasen dominiert der Aufschlag, Sätze gehen häufiger in den Tiebreak, und ein Außenseiter mit starkem Service kann so auch gegen einen deutlich besser platzierten Gegner standhalten.

Eine Frage bleibt offen: Sand wird nur bei den French Open gespielt, Rasen nur in Wimbledon. Belag und Turnier lassen sich nicht trennen, der gemessene Effekt könnte genauso gut ein Turnier-Effekt sein.

Ein Teil davon ist prüfbar. Hartplatz umfasst zwei Turniere. Wäre der Belag die Ursache, müssten Australian Open und US Open eng beieinander liegen.

In [ ]:
h2_turnier = quote_je_gruppe(df_gs, 'Tournament')
h2_turnier['Belag'] = df_gs.groupby('Tournament')['Surface'].first()
print(h2_turnier)

# Belegt die Aussage ob jedes Turnier genau einen Belag hat.
print("\nHat jedes Turnier genau einen Belag?",
      bool((df_gs.groupby('Tournament')['Surface'].nunique() == 1).all()))

hart = h2_turnier.loc[['Australian Open', 'US Open'], 'Quote']
print("Unterschied der beiden Hartplatz-Turniere:", round(hart.max() - hart.min(), 2), "Prozentpunkte")
print("Spannweite zwischen den drei Belaegen:    ", round(h2['Quote'].max() - h2['Quote'].min(), 2), "Prozentpunkte")

Sie liegen nicht eng beieinander. Zwischen den beiden Hartplatz-Turnieren liegen 2,53 Prozentpunkte, fast genau so viel wie zwischen allen drei Belägen (Spannweite der drei Werte = 2,58). Der Unterschied lässt sich also nicht dem Belag zuschreiben, ebenso plausibel sind der Zeitpunkt in der Saison, die Reisebelastung oder das Teilnehmerfeld. Trennen lassen sich die beiden Einflüsse in diesen Daten nicht.

**H2 ist nicht bestätigt.** Der Effekt ist zwar signifikant, aber mit Cramérs V von 0,023 ohne praktische Bedeutung und nicht sauber vom Turnier zu trennen. Und die Richtung ist widerlegt: nicht Sand, sondern Rasen hat die niedrigste Quote.


<a id="kap3-3"></a>

### 3.3 Vorhersagbarkeit im Zeitverlauf (H3)

Die 3. Hypothese hat zwei Teile:

- Erstens soll die Favoritensiegquote über die Jahre gestiegen sein.
- Zweitens soll die Big-Three-Ära besonders vorhersagbar gewesen sein, also die Jahre, in denen Federer, Nadal und Djokovic die Turniere unter sich ausgemacht haben.

Beides wird getrennt geprüft, zuerst die Quote pro Jahr, danach der Vergleich der Zeiträume. Zwei Jahre sind unvollständig. 2020 fiel Wimbledon wegen der Pandemie aus, 2026 fehlt das US Open, weil es zum Zeitpunkt der Analyse noch nicht gespielt war. Beide Jahre beruhen auf drei statt vier Turnieren und werden im Diagramm gekennzeichnet.

In [ ]:
h3 = quote_je_gruppe(df_gs, 'Year')
h3['Turniere'] = df_gs.groupby('Year')['Tournament'].nunique()
print(h3.to_string())

In [ ]:
gesamt = df_gs['Fav_Rang_Sieg'].mean() * 100

plt.figure(figsize=(10, 5))
plt.axvspan(2004, 2022, color='gold', alpha=0.15, label='Big-Three-Ära (2004 bis 2022)')
plt.plot(h3.index, h3['Quote'], marker='o', lw=2, label='Favoritensiegquote')
plt.axhline(gesamt, ls=':', color='darkred', lw=1.5, label=f'Gesamtquote ({gesamt:.1f} %)')

# Jahre mit nur drei statt vier Turnieren hervorheben
unvollstaendig = h3[h3['Turniere'] < 4]
plt.scatter(unvollstaendig.index, unvollstaendig['Quote'], s=130, facecolors='none',
            edgecolors='red', lw=1.5, zorder=5, label='unvollständiges Jahr')

plt.xlabel("Jahr")
plt.ylabel("Favoritensiegquote in %")
plt.title("Favoritensiegquote je Jahr")
plt.ylim(60, 80)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

Ein Anstieg über die Jahre ist nicht zu erkennen. Die Kurve steigt bis etwa 2015, fällt danach wieder ab und liegt zuletzt ungefähr dort, wo sie 2000 begonnen hat. Der Verlauf ähnelt eher einem Bogen statt einem wirklichen Trend.

Der Bogen deckt sich auffällig mit dem markierten Bereich. Ob dieser optische Eindruck auch als Zahl standhält, prüft der nächste Schritt:
Die drei Zeiträume im direkten Vergleich, dazu die Rangkorrelation zwischen Jahr und Quote für den ersten Teil der Hypothese.

In [ ]:
grenzen_aera = [1999, 2003, 2022, 2026]
labels_aera  = ["vor 2004", "2004-2022", "ab 2023"]
df_gs['Aera'] = pd.cut(df_gs['Year'], bins=grenzen_aera, labels=labels_aera)

h3_aera = quote_je_gruppe(df_gs, 'Aera')
print(h3_aera)

Tabelle_aera = pd.crosstab(df_gs['Aera'], df_gs['Fav_Rang_Sieg'])
chi2_a, p_a, dof_a, erwartet_a = chi2_contingency(Tabelle_aera)

print("\nChi-Quadrat:", round(chi2_a, 3), "| p-Wert:", round(p_a, 6), "| Freiheitsgrade:", dof_a)
print("Kleinste erwartete Haeufigkeit:", round(erwartet_a.min(), 1))
print("Cramers V:", round(np.sqrt(chi2_a / Tabelle_aera.values.sum()), 4))

# Erster Teil der Hypothese: steigt die Quote ueber die Jahre?
rho_jahr = h3['Quote'].corr(pd.Series(h3.index, index=h3.index), method='spearman')
print("\nSpearman-Korrelation Jahr ~ Quote:", round(rho_jahr, 3))

**Der zweite Teil der Hypothese stimmt.** Die Big-Three-Ära liegt rund vier Prozentpunkte über beiden Vergleichszeiträumen, und mit p = 0,00003 ist das kein Zufallsergebnis. Cramérs V beträgt 0,040, also weiterhin klein, aber knapp doppelt so groß wie der Belag-Effekt aus 3.2.

**Der erste Teil stimmt nicht.** Die Rangkorrelation zwischen Jahr und Quote liegt bei 0,03, praktisch null. Daher gibt es keinen wirklichen Anstieg über die Jahre, denn vor und nach der Ära liegt die Quote auf fast identischem Niveau.

**H3 ist damit teilweise bestätigt.** Die Ära war messbar vorhersagbarer, gestiegen ist die Vorhersagbarkeit aber nicht.

**Einschränkung:** Der Zeitraum vor 2004 umfasst nur vier Jahre, die Vergleichsbasis ist also schmal. Und die Grenze bei 2022 ist eine Setzung, denn Djokovic gewann 2023 noch drei der vier Turniere. Zieht man sie stattdessen bei 2023, ändert sich am Befund nichts: Die Ära kommt dann auf 72,40 % gegenüber 68,46 % davor und 69,71 % danach, bei p = 0,0006.

<a id="kap3-4"></a>

### 3.4 Top-10-Dominanz (H4)

Geprüft wird, ob Top-10-Spieler überproportional oft gegen Gegner außerhalb der Top 10 gewinnen. Aussagekräftig sind dafür nur die Matches, in denen genau einer der beiden zu den Top 10 gehört. Treffen zwei Top-10-Spieler oder zwei Außenseiter aufeinander, gibt es nichts zu vergleichen. Diese Auswahl wurde bereits in Abschnitt 2.5 mit der Hilfsspalte als `Top10_Sieg` angelegt.

Neben der Gesamtquote wird nach Turnierrunde aufgeschlüsselt, denn die Gegner werden mit jeder Runde stärker. Dafür wird vorher auf genau diese Duelle gefiltert, sonst zählt `size` in der Hilfsfunktion alle Matches einer Runde mit und die Fallzahl fällt viel zu hoch aus.

In [ ]:
# Nur Matches mit genau einem Top-10-Spieler. 
# Ohne diesen Filter wuerde quote_je_gruppe mit size auch die uebrigen Matches der Runde mitzaehlen.
duelle = df_gs[df_gs['Top10_Sieg'].notna()]

print("Duelle Top 10 gegen Nicht-Top-10:", len(duelle))
print("Siegquote des Top-10-Spielers:   ", round(duelle['Top10_Sieg'].mean() * 100, 2), "%")
print("Favoritensiegquote insgesamt:    ", round(df_gs['Fav_Rang_Sieg'].mean() * 100, 2), "%")

# Reihenfolge vorgeben, groupby wuerde alphabetisch sortieren
runden = ['1st Round', '2nd Round', '3rd Round', '4th Round',
          'Quarterfinals', 'Semifinals', 'The Final']
h4 = quote_je_gruppe(duelle, 'Round', ziel='Top10_Sieg').reindex(runden)
print("\n", h4, sep="")

In [ ]:
gesamt = df_gs['Fav_Rang_Sieg'].mean() * 100
beschriftung = ['1. Runde', '2. Runde', '3. Runde', 'Achtelfinale',
                'Viertelfinale', 'Halbfinale', 'Finale']

plt.figure(figsize=(9, 5))
plt.bar(range(len(h4)), h4['Quote'], width=0.65, color='#4a7ebb')
plt.axhline(gesamt, ls=':', color='darkred', lw=1.5,
            label=f'Favoritensiegquote insgesamt ({gesamt:.1f} %)')

# Fallzahl in jeden Balken schreiben
for i, n in enumerate(h4['Matches']):
    plt.text(i, 62, f"n={n}", ha='center', color='white', fontsize=9)

plt.xticks(range(len(h4)), beschriftung, rotation=20)
plt.ylabel("Siegquote des Top-10-Spielers in %")
plt.title("Top 10 gegen Nicht-Top-10, je Runde")
plt.ylim(60, 95)
plt.grid(axis='y', alpha=0.3)
plt.legend()
plt.show()

Die Dominanz nimmt mit fortschreitender Runde ab, von gut 90 % in der ersten Runde auf rund 80 % ab dem Achtelfinale. Das ist plausibel, denn wer als Nicht-Top-10-Spieler die dritte Runde übersteht, hat bereits starke Gegner geschlagen. Halbfinale und Finale beruhen auf 85 und 22 Matches, dort verschiebt ein einzelnes Match die Quote um mehr als einen Prozentpunkt. Daher sollte der Anstieg im Halbfinale nicht überbewertet werden.

**Jedoch bleibt ein Einwand offen:** In diesen Duellen ist der Rangunterschied naturgemäß groß, und aus 3.1 ist bekannt, dass die Quote mit der Rangdifferenz steigt. Eventuell misst H4 gar nichts Eigenes, sondern denselben Effekt noch einmal nur in einer anderen "Verpackung". Der Test dafür: 
dieselbe Rangdifferenz-Klasse, einmal mit und einmal ohne Top-10-Beteiligung.

In [ ]:
# In diesen Duellen ist der Top-10-Spieler immer zugleich der Rang-Favorit,
# deshalb lassen sich beide Gruppen ueber Fav_Rang_Sieg vergleichen.
print("Top-10-Spieler ist immer der Rang-Favorit:",
      bool((duelle['Top10_Sieg'] == duelle['Fav_Rang_Sieg']).all()))

df_gs['Top10_Duell'] = df_gs['Top10_Sieg'].notna()

vergleich = (df_gs.groupby(['Rang_Klasse', 'Top10_Duell'], observed=True)['Fav_Rang_Sieg']
                  .agg(Quote='mean', Matches='size'))
vergleich['Quote'] = (vergleich['Quote'] * 100).round(2)
print("\n", vergleich, sep="")

Der Einwand entkräftet sich. Bei gleicher Rangdifferenz gewinnt der Favorit **deutlich häufiger**, wenn er zu den **Top 10** gehört: in der Klasse 11 bis 20 sind es 80,00 % statt 56,39 %, in der Klasse 21 bis 50 dann 85,31 % statt 63,25 %. In den größten vier Klassen liegt der Abstand zwischen 16 und 24 Prozentpunkten. Der Top-10-Status trägt also eigene Information, er ist nicht bloß eine andere Schreibweise für einen großen Rangabstand.

Eine Ausnahme bildet die Klasse 1 bis 5 mit 56,86 % gegenüber 57,18 %, praktisch identisch. Dort steht ein Top-10-Spieler einem direkten Ranglistennachbarn gegenüber, und mit 51 Matches ist die Gruppe verhältnismäßig klein.

**H4 ist bestätigt.** Top-10-Spieler gewinnen 86,02 % ihrer Matches gegen Gegner außerhalb der Top 10, gegenüber 71,53 % Favoritensiegquote insgesamt.

Das beantwortet die offene Frage aus 3.1. Dort blieb unklar, ob zehn Ränge an der Spitze dasselbe bedeuten wie im Mittelfeld. Sie bedeuten deutlich mehr, auch in ihrer Gewichtung.

<a id="kap3-5"></a>

### 3.5 Prädiktorvergleich: Weltrangliste vs. Buchmacherquote (H5)

Diese Hypothese wurde gegenüber dem Proposal geändert, siehe Abschnitt 1.1. Ursprünglich sollte die ATP-Punktedifferenz gegen die Rangdifferenz antreten. Der Abschnitt beginnt deshalb mit der Prüfung dieser ersten Fassung, bevor er zur geänderten übergeht.

Schritt 1: Benennen Rang und Punkte überhaupt jemals verschiedene Favoriten? Schritt 2: Wie schlägt sich die Weltrangliste gegen die Buchmacherquote, gemessen daran, wer häufiger den Sieger benennt?

Beide Trefferquoten müssen dabei auf derselben Match-Auswahl berechnet werden. Quoten fehlen in einigen Matches, und würde man den Rang-Favoriten über alle 12.871 Matches rechnen, den Quoten-Favoriten aber nur über die mit Quote, wäre der Vergleich verzerrt.

In [ ]:
# Punkte-Favorit ableiten, mit derselben Absicherung wie in 2.5
pts_ok = df_gs[['Pts_1','Pts_2']].notna().all(axis=1) & (df_gs['Pts_1'] != df_gs['Pts_2'])
fav_pts = pd.Series(np.where(df_gs['Pts_1'] > df_gs['Pts_2'],   # mehr Punkte = Favorit
                             df_gs['Player_1'], df_gs['Player_2']), index=df_gs.index)
df_gs['Fav_Pts'] = fav_pts.where(pts_ok)

# Nur Matches, in denen beide Praediktoren einen Favoriten benennen
beide = pts_ok & df_gs['Fav_Rang'].notna()
print("Vergleichbare Matches:", int(beide.sum()), "von", len(df_gs))
print("  ohne Punktangabe:   ", int(df_gs[['Pts_1','Pts_2']].isna().any(axis=1).sum()))
print("  Punkte-Gleichstand: ", int((df_gs['Pts_1'] == df_gs['Pts_2']).sum()))

uneinig_pts = beide & (df_gs['Fav_Rang'] != df_gs['Fav_Pts'])
print("\nRang und Punkte benennen einen anderen Favoriten:",
      int(uneinig_pts.sum()), "von", int(beide.sum()), "Matches")

In keinem einzigen der vergleichbaren Matches benennen Rang und Punkte einen unterschiedlichen Favoriten. Das ist kein Zufall, sondern folgt aus der Konstruktion: Die Weltrangliste wird direkt aus den ATP-Punkten gebildet, wer mehr Punkte hat, steht weiter oben. Schon in 2.4 lag die Spearman-Korrelation zwischen beiden bei -0,98.

Damit ist die ursprüngliche Hypothese des Proposals nicht prüfbar. Zwei Prädiktoren, die immer denselben Spieler benennen, können sich in der Trefferquote nicht unterscheiden, es gäbe schlicht nichts zu messen. Die fehlenden 2.806 Matches erklären sich aus den Jahren 2000 bis 2004 (2005 rund 75 %), für die keine Punkte vorliegen, plus fünf Gleichständen.

Als Gegenpart zur Weltrangliste tritt deshalb die Buchmacherquote an. Sie entsteht unabhängig von der Rangliste und verarbeitet zusätzlich Form, Verletzungen und direkte Vergleiche.

In [ ]:
#  gemeinsame Basis, nur Matches mit Rang- UND Quoten-Favorit
gemeinsam = df_gs[df_gs['Fav_Rang'].notna() & df_gs['Fav_Quote'].notna()]
print("Gemeinsame Basis:", len(gemeinsam), "von", len(df_gs), "Matches")

treffer_rang  = gemeinsam['Fav_Rang_Sieg'].mean() * 100
treffer_quote = gemeinsam['Fav_Quote_Sieg'].mean() * 100
print("\nTrefferquote Rang-Favorit  :", round(treffer_rang, 2), "%")
print("Trefferquote Quoten-Favorit:", round(treffer_quote, 2), "%")

# Die Matches, in denen beide einen anderen Favoriten benennen
uneinig = gemeinsam[gemeinsam['Fav_Rang'] != gemeinsam['Fav_Quote']]
print("\nUneinige Matches:", len(uneinig),
      "=", round(len(uneinig) / len(gemeinsam) * 100, 2), "%")
print("  davon gewinnt der Quoten-Favorit:", round(uneinig['Fav_Quote_Sieg'].mean() * 100, 2), "%")
print("  davon gewinnt der Rang-Favorit:  ", round(uneinig['Fav_Rang_Sieg'].mean() * 100, 2), "%")

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(['Rang-Favorit', 'Quoten-Favorit'], [treffer_rang, treffer_quote],
        width=0.5, color=['#4a7ebb', '#c98a5e'])
plt.axhline(50, ls='--', color='grey', lw=1, label='Münzwurf (50 %)')

for i, wert in enumerate([treffer_rang, treffer_quote]):
    plt.text(i, wert + 1.5, f"{wert:.2f} %", ha='center')

plt.ylabel("Trefferquote in %")
plt.title(f"Wer benennt häufiger den Sieger? ({len(gemeinsam)} Matches)")
plt.ylim(45, 85)
plt.grid(axis='y', alpha=0.3)
plt.legend()
plt.show()

**H5 ist bestätigt.** Die Buchmacherquote trifft in 76,64 % der Matches, die Weltrangliste in 71,99 %, ein Vorsprung von 4,65 Prozentpunkten auf identischer Datenbasis. Der kleine Unterschied zur Zahl aus 2.5 (71,53 %) kommt daher, dass dort alle 12.871 Matches eingehen, hier nur die 12.089 mit gültiger Quote.

Aufschlussreich sind die 1.832 Matches, in denen beide einen anderen Favoriten benennen. Dort gewinnt der Quoten-Favorit in 65,34 % der Fälle. Widersprechen die Buchmacher der Rangliste, behalten sie also in zwei von drei Fällen recht. Diese gut 15 % der Matches machen den gesamten Vorsprung aus, in den übrigen 85 % sagen beide ohnehin dasselbe voraus.

Das ist plausibel, denn die Quote entsteht später und weiß mehr. Die Weltrangliste bildet die Ergebnisse der zurückliegenden 52 Wochen ab, die Quote kennt zusätzlich die aktuelle Form, Verletzungen, den Belag und die direkte Bilanz beider Spieler. Sie ist kein besseres Maß für Spielstärke, sondern schlicht aktueller.

Eine Einschränkung gehört dazu: Buchmacherquoten enthalten eine Marge und spiegeln auch das Wettverhalten des Publikums, sie sind also keine reinen Wahrscheinlichkeiten. Für die hier gestellte Frage, wer häufiger den Sieger benennt, spielt das keine Rolle.

<a id="kap4"></a>

## 4. Vorhersage des Matchausgangs

[&#8593; Zurück zum Inhaltsverzeichnis](#toc)

Die explorative Analyse hat gezeigt, welche Merkmale mit dem Matchausgang zusammenhängen. Jetzt die umgekehrte Frage: Lässt sich daraus eine Vorhersage bauen, die besser ist als die einfachste denkbare Regel?

Vorhergesagt wird `Fav_Rang_Sieg`, also ob der besser platzierte Spieler gewinnt. Ziel und Merkmale sind bewusst symmetrisch gebaut, also Rang des Favoriten und Abstand zum Gegner statt Spieler 1 und Spieler 2. Sonst lernte das Modell die Reihenfolge der Spalten mit, und die ist im Datensatz willkürlich.

Als Messlatte dienen zwei Baselines auf denselben Testdaten, wobei nach unten die triviale Regel "immer gewinnt der Rang-Favorit", nach oben der Quoten-Favorit aus 3.5 verwendet werden. Die Buchmacherquote selbst ist dabei kein Merkmal, sie ist der Vergleichsmaßstab und nicht der Rohstoff.

**Warum eine logistische Regression?** Die Zielvariable ist binär, der Favorit gewinnt oder er gewinnt nicht. Das ist eine Klassifikationsaufgabe, und die logistische Regression ist dafür das einfachste etablierte Verfahren.

Ausschlaggebend ist hier aber ihre Interpretierbarkeit. Die Koeffizienten sagen direkt, welches Merkmal wie stark und in welche Richtung wirkt. Damit lässt sich prüfen, ob das Modell dieselben Zusammenhänge findet wie die explorative Analyse in Kapitel 3. 

In [ ]:
merkmale = df_gs.copy()
merkmale['Rang_Fav']    = merkmale[['Rank_1','Rank_2']].min(axis=1)   # der bessere Rang
merkmale['Top10_Fav']   = (merkmale['Rang_Fav'] <= 10).astype(int)
merkmale['Top10_Beide'] = (merkmale[['Rank_1','Rank_2']].max(axis=1) <= 10).astype(int)

# Belag und Runde sind Text, get_dummies macht daraus 0/1-Spalten
X = pd.get_dummies(
    merkmale[['Rang_Fav', 'Rang_Diff', 'Top10_Fav', 'Top10_Beide',
              'Year', 'Surface', 'Round']],
    columns=['Surface', 'Round'], drop_first=True).astype(float)
y = merkmale['Fav_Rang_Sieg'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

# Baseline nach unten: immer auf den Rang-Favoriten tippen, das ist sein Siegeranteil im Test.
# Baseline nach oben: der Quoten-Favorit, gemessen auf denselben Testmatches.
baseline_rang  = y_test.mean() * 100
test_roh       = df_gs.loc[y_test.index]
mit_quote      = test_roh['Fav_Quote_Sieg'].notna()
baseline_quote = test_roh.loc[mit_quote, 'Fav_Quote_Sieg'].mean() * 100

print("Merkmale:", X.shape[1], "| Training:", len(y_train), "| Test:", len(y_test))
print("Baseline 'immer der Rang-Favorit'  :", round(baseline_rang, 2), "%")
print("Baseline 'immer der Quoten-Favorit':", round(baseline_quote, 2), "%",
      f"(auf {int(mit_quote.sum())} von {len(test_roh)} Testmatches)")

In [ ]:
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
logreg.fit(X_train, y_train)
# Trainings- gegen Testgenauigkeit: liegt Training deutlich hoeher, hat das Modell auswendig gelernt
acc_train  = accuracy_score(y_train, logreg.predict(X_train)) * 100
acc_logreg = accuracy_score(y_test,  logreg.predict(X_test))  * 100
print("Genauigkeit auf den Trainingsdaten:", round(acc_train, 2), "%")
print("Genauigkeit auf den Testdaten     :", round(acc_logreg, 2), "%")

# Welche Merkmale zieht das Modell heran? Skaliert, also direkt vergleichbar.
gewichte = pd.Series(logreg[-1].coef_[0], index=X.columns).sort_values(key=abs, ascending=False)
print("\nStaerkste Koeffizienten:")
print(gewichte.head(6).round(3))

**Das Modell schlägt die Baselines nicht.** Mit gut 71 % liegt es praktisch genau dort, wo auch "immer gewinnt der Rang-Favorit" liegt. Aus 13 Merkmalen und knapp 10.000 Trainingsmatches entsteht also keine bessere Trefferquote.

In [ ]:
namen = ['Baseline\nRang-Favorit', 'Logistische\nRegression', 'Buchmacher\nQuoten-Favorit']
werte = [baseline_rang, acc_logreg, baseline_quote]

plt.figure(figsize=(8, 5))
plt.bar(namen, werte, color=['#999999', '#4a7ebb', '#c98a5e'], width=0.55)
plt.axhline(baseline_rang, ls=':', color='grey', lw=1.5)

for i, wert in enumerate(werte):
    plt.text(i, wert + 0.4, f"{wert:.2f} %", ha='center')

plt.ylabel("Trefferquote auf den Testdaten in %")
plt.title("Modell gegen Baselines")
plt.ylim(65, 82)
plt.grid(axis='y', alpha=0.3)
plt.show()

Auf den Trainingsdaten erreicht das Modell 71,52 %, auf den Testdaten 71,41 %, also praktisch dasselbe. Auswendig gelernt hat es die Trainingsdaten damit nicht. Es scheitert also nicht daran, dass es sie zu genau nachbildet, sondern daran, dass in den Merkmalen für eine bessere Entscheidung nichts drinsteckt.

Die Koeffizienten zeigen immerhin, dass das Modell die richtigen Dinge heranzieht: Top-10-Status des Favoriten, dann sein absoluter Rang, dann die Rangdifferenz, also genau die Reihenfolge aus 3.4 und 3.1. Der Belag taucht unter den stärksten Merkmalen nicht auf, was zu H2 passt.

**Die Grenze liegt bei den Merkmalen, nicht beim Verfahren.** Alle Merkmale stammen aus der Weltrangliste, und die steckt bereits vollständig in der Baseline. Ein Modell kann daraus nichts Neues gewinnen. Der Quoten-Favorit erreicht auf denselben Testdaten 77,58 %, weil Buchmacher einpreisen, was in keiner Ranglistenspalte steht: Form, Verletzungen, direkte Bilanz.

Damit schließt sich der Bogen zu H5. Der dort gemessene Rückstand der Weltrangliste gegenüber der Quote ist mit Modellierung allein nicht aufzuholen, es fehlen schlicht die passenden Merkmale.

<a id="kap5"></a>

## 5. Fazit und Limitationen

[&#8593; Zurück zum Inhaltsverzeichnis](#toc)

### Ergebnis je Hypothese

| | Hypothese | Ergebnis | Kernzahl |
|---|---|---|---|
| H1 | Rangdifferenz | bestätigt | 57,16 % bei 1 bis 5 Rängen Abstand, 79,71 % bei über 100 |
| H2 | Belag | nicht bestätigt | signifikant (p = 0,031), aber Cramérs V nur 0,023, und Rasen statt Sand |
| H3 | Zeitverlauf | teilweise bestätigt | Ära 72,69 % gegenüber 68,46 % davor und 68,98 % danach, aber kein Anstieg |
| H4 | Top-10-Dominanz | bestätigt | 86,02 % gegenüber 71,53 % insgesamt |
| H5 | Prädiktorvergleich | bestätigt | Buchmacher 76,64 %, Weltrangliste 71,99 % |

Über alle fünf hinweg zeichnet sich dasselbe Bild. Die Weltrangliste sagt den Ausgang zuverlässig voraus, wenn der Abstand groß ist, und ist bei eng benachbarten Spielern kaum mehr wert als ein Münzwurf. Was das Match umgibt, trägt dagegen fast nichts bei: Zwischen dem besten und dem schlechtesten Belag liegen 2,58 Prozentpunkte, zwischen den Ären rund vier.

Etwas hinzu kommt nur dort, wo Information steckt, die in der Rangposition selbst nicht enthalten ist. Der Top-10-Status ist so ein Fall, denn er wirkt auch dann noch, wenn die Rangdifferenz konstant gehalten wird. Die Buchmacherquote ist der zweite, weil sie aktuelle Form, Verletzungen und direkte Bilanzen beinhaltet.

Methodisch ist H2 der lehrreichste Abschnitt. Der Test fällt signifikant aus, der Effekt ist trotzdem bedeutungslos. Bei 12.871 Matches wird auch der kleinste Unterschied statistisch auffällig, ein p-Wert ohne Maß für die Effektstärke sagt deshalb wenig.

### Vorhersage

Das Modell aus Kapitel 4 erreicht 71,41 % und bleibt damit sogar knapp unter der trivialen Regel, die auf denselben Testdaten 71,54 % trifft. Aus 13 Merkmalen und knapp 10.000 Trainingsmatches entsteht also keine bessere Vorhersage als aus einem einzigen Satz Regel.

Das liegt nicht am Verfahren, sondern an den Merkmalen. Alle stammen aus der Weltrangliste, und die steckt bereits vollständig in der Baseline. Der Quoten-Favorit kommt auf denselben Testdaten auf 77,58 %, weil Buchmacher Form, Verletzungen und direkte Bilanzen einpreisen. Genau dort läge der nächste Schritt, mit den vorhandenen Spalten ist die Grenze erreicht.

### Limitationen

- **Nur Herren-Einzel der ATP.** Damen- und Mixed-Wettbewerbe sind nicht enthalten, die Ergebnisse gelten nicht automatisch für sie.
- **Belag und Turnier sind nicht trennbar.** Sand kommt nur bei den French Open vor, Rasen nur in Wimbledon. Was als Belag-Effekt gemessen wird, kann genauso ein Turnier-Effekt sein. Abschnitt 3.2 zeigt das an den beiden Hartplatz-Turnieren.
- **Zwei unvollständige Jahre.** 2020 fiel Wimbledon wegen der Pandemie aus, 2026 fehlt das US Open. Beide beruhen auf drei statt vier Turnieren.
- **ATP-Punkte erst ab 2006.** Für 2000 bis 2005 fehlen sie ganz oder überwiegend, zeitbezogene Auswertungen laufen deshalb über den Rang.
- **Fehlende Buchmacherquoten.** In 782 von 12.871 Matches lässt sich kein Quoten-Favorit bestimmen, überwiegend im Jahr 2000. Der Vergleich in 3.5 beruht daher auf 12.089 Matches.
- **Die Ären-Grenze ist eine Setzung.** Djokovic gewann 2023 noch drei der vier Turniere. Eine Verschiebung der Grenze auf 2023 ändert den Befund zwar nicht, die Einteilung bleibt aber eine Entscheidung und kein Messwert.
- **23 Matches sind fälschlich als "Best of 3" kodiert**, obwohl Grand Slams über fünf Sätze gehen. Für die untersuchten Fragen ist das folgenlos, es zeigt aber, dass der Datensatz nicht fehlerfrei ist.
- **Alle Befunde sind korrelativ.** Sie beschreiben Zusammenhänge, keine Ursachen. Warum Favoriten auf Rasen seltener gewinnen, lässt sich aus Beobachtungsdaten ohne Eingriff nicht beantworten.
- **Das Vorhersagemodell kennt nur die Weltrangliste.** Form, Verletzungen und direkte Bilanzen stehen nicht im Datensatz. Kapitel 4 zeigt deshalb die Grenze dieser Merkmale, nicht die Grenze des Verfahrens.